# Data Mining Assignment: Spark Data Lake Preprocessing

#Name:Ansh Bhargava
#Roll No:LCB2023012

In [1]:
from pathlib import Path
import shutil

from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder
         .appName("SalesDataLakePreprocessing")
         .master("local[*]")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.sql.ansi.enabled", "false")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 4.0.4


In [2]:
data_lake = Path("data_lake")
bronze_dir = data_lake / "bronze"
silver_dir = data_lake / "silver" / "preprocessed_sales"
bronze_dir.mkdir(parents=True, exist_ok=True)

source_path = Path("/content/sales.csv")

if not source_path.exists():
    raise FileNotFoundError("Upload sales.csv to /content/.")

bronze_path = bronze_dir / "sales.csv"
shutil.copyfile(source_path, bronze_path)

assert source_path.read_bytes() == bronze_path.read_bytes()
print("Bronze file:", bronze_path)
print("Byte-for-byte copy verified:", source_path.stat().st_size, "bytes")


Bronze file: data_lake/bronze/sales.csv
Byte-for-byte copy verified: 109537 bytes


## 1. Load and inspect the data

In [3]:
raw = (spark.read
       .option("header", True)
       .option("inferSchema", False)
       .csv(str(bronze_path)))

raw.printSchema()
raw.show(10, truncate=False)
before_count = raw.count()
print("Bronze row count:", before_count)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+------------

## 2. Identify data-quality discrepancies with Spark


In [4]:
duplicate_count = before_count - raw.dropDuplicates().count()
print("Exact duplicate rows:", duplicate_count)

missing_counts = raw.select([
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
    for c in raw.columns
])
print("Null/blank values by column:")
missing_counts.show(truncate=False)

Exact duplicate rows: 10
Null/blank values by column:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [5]:
text_columns = [
    "customer_name", "product", "category", "payment_method",
    "city", "state", "order_status"
]

whitespace_issues = raw.select([
    F.sum(F.when(F.col(c) != F.trim(F.col(c)), 1).otherwise(0)).alias(c)
    for c in text_columns
])
print("Rows with extra surrounding whitespace by text column:")
whitespace_issues.show(truncate=False)

print("Case variants representing the same normalized text:")
for c in text_columns:
    variants = (raw
        .filter(F.col(c).isNotNull() & (F.trim(F.col(c)) != ""))
        .groupBy(F.lower(F.trim(F.col(c))).alias("normalized"))
        .agg(
            F.countDistinct(F.trim(F.col(c))).alias("variant_count"),
            F.sort_array(F.collect_set(F.trim(F.col(c)))).alias("raw_variants")
        )
        .filter(F.col("variant_count") > 1))
    if variants.count() > 0:
        print(f"Column: {c}")
        variants.show(truncate=False)

Rows with extra surrounding whitespace by text column:
+-------------+-------+--------+--------------+----+-----+------------+
|customer_name|product|category|payment_method|city|state|order_status|
+-------------+-------+--------+--------------+----+-----+------------+
|3            |0      |0       |0             |3   |0    |0           |
+-------------+-------+--------+--------------+----+-----+------------+

Case variants representing the same normalized text:
Column: category
+-----------+-------------+--------------------------+
|normalized |variant_count|raw_variants              |
+-----------+-------------+--------------------------+
|beauty     |2            |[Beauty, beauty]          |
|electronics|2            |[ELECTRONICS, Electronics]|
|sports     |3            |[SPORTS, Sports, sports]  |
|clothing   |2            |[CLOTHING, Clothing]      |
+-----------+-------------+--------------------------+

Column: city
+----------+-------------+------------------------+
|normali

In [6]:
numeric_checks = raw.select(
    F.sum(F.when(F.col("quantity").cast("double").isNull() | (F.col("quantity").cast("double") <= 0), 1).otherwise(0)).alias("invalid_quantity"),
    F.sum(F.when(F.col("unit_price").cast("double").isNull() | (F.col("unit_price").cast("double") <= 0), 1).otherwise(0)).alias("invalid_unit_price"),
    F.sum(F.when(
        F.col("discount_percent").cast("double").isNull() |
        (F.col("discount_percent").cast("double") < 0) |
        (F.col("discount_percent").cast("double") > 100), 1
    ).otherwise(0)).alias("invalid_discount_percent")
)
print("Invalid numerical values:")
numeric_checks.show(truncate=False)

parsed_date = F.coalesce(
    F.to_date(F.trim("order_date"), "yyyy-MM-dd"),
    F.to_date(F.trim("order_date"), "yyyy/MM/dd"),
    F.to_date(F.trim("order_date"), "MM-dd-yyyy"),
    F.to_date(F.trim("order_date"), "dd-MM-yyyy"),
)
invalid_date_count = raw.filter(
    F.col("order_date").isNull() |
    (F.trim("order_date") == "") |
    parsed_date.isNull()
).count()
print("Invalid/unparseable dates:", invalid_date_count)
raw.filter(parsed_date.isNull()).select("order_id", "order_date").show(truncate=False)

Invalid numerical values:
+----------------+------------------+------------------------+
|invalid_quantity|invalid_unit_price|invalid_discount_percent|
+----------------+------------------+------------------------+
|12              |8                 |8                       |
+----------------+------------------+------------------------+

Invalid/unparseable dates: 4
+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |31/02/2026|
|100448  |not-a-date|
|100634  |2025-02-30|
|100637  |2026-13-05|
+--------+----------+



## 3. Basic preprocessing for the Silver layer

In [7]:
clean = raw.dropDuplicates()


for c in raw.columns:
    clean = clean.withColumn(c, F.trim(F.col(c)))

clean = clean.select([
    F.when(F.col(c) == "", None).otherwise(F.col(c)).alias(c)
    for c in clean.columns
])

clean = clean.withColumn("customer_id", F.upper("customer_id"))
for c in ["customer_name", "category", "city", "state", "order_status"]:
    clean = clean.withColumn(c, F.initcap(F.lower(F.col(c))))

clean = clean.withColumn(
    "payment_method",
    F.when(F.lower("payment_method") == "upi", F.lit("UPI"))
     .otherwise(F.col("payment_method"))
)

clean = clean.fillna({
    "customer_name": "Unknown",
    "product": "Unknown",
    "category": "Unknown",
    "payment_method": "Unknown",
    "city": "Unknown",
    "state": "Unknown",
    "order_status": "Unknown",
})

clean = clean.filter(F.col("order_id").isNotNull() & F.col("customer_id").isNotNull())

In [8]:
clean = (clean
    .withColumn("quantity_num", F.col("quantity").cast("double"))
    .withColumn("unit_price_num", F.col("unit_price").cast("double"))
    .withColumn("discount_num", F.col("discount_percent").cast("double")))

clean = (clean
    .withColumn("quantity_num", F.when(F.col("quantity_num") > 0, F.col("quantity_num")))
    .withColumn("unit_price_num", F.when(F.col("unit_price_num") > 0, F.col("unit_price_num")))
    .withColumn(
        "discount_num",
        F.when(F.col("discount_num").between(0, 100), F.col("discount_num"))
    ))

numeric_medians = {
    c: clean.approxQuantile(c, [0.5], 0.0)[0]
    for c in ["quantity_num", "unit_price_num", "discount_num"]
}
print("Medians used for invalid/missing numerical values:", numeric_medians)

clean = (clean
    .fillna(numeric_medians)
    .drop("quantity", "unit_price", "discount_percent")
    .withColumnRenamed("quantity_num", "quantity")
    .withColumnRenamed("unit_price_num", "unit_price")
    .withColumnRenamed("discount_num", "discount_percent")
    .withColumn("quantity", F.col("quantity").cast("integer"))
    .withColumn("unit_price", F.col("unit_price").cast(T.DecimalType(12, 2)))
    .withColumn("discount_percent", F.col("discount_percent").cast(T.DecimalType(5, 2)))
    .withColumn("order_date", parsed_date))

clean = clean.select(
    "order_id", "customer_id", "customer_name", "product", "category",
    "quantity", "unit_price", "discount_percent", "payment_method",
    "city", "state", "order_date", "order_status"
)

clean.printSchema()
clean.show(10, truncate=False)

Medians used for invalid/missing numerical values: {'quantity_num': 5.0, 'unit_price_num': 1850.0, 'discount_num': 20.0}
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(12,2) (nullable = true)
 |-- discount_percent: decimal(5,2) (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = false)

+--------+-----------+--------------+----------------------+-----------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|order_id|customer_id|customer_name |product               |category   |quantity|unit_price|discount_percent|payment_method  |city     |s

## 4. Write and verify the Silver layer

In [9]:
silver_dir = Path("data_lake/silver/preprocessed_sales")
temp_dir = Path("data_lake/silver/temp_output")

if silver_dir.exists():
    shutil.rmtree(silver_dir)

if temp_dir.exists():
    shutil.rmtree(temp_dir)

(clean.coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .csv(str(temp_dir)))

silver_dir.mkdir(parents=True, exist_ok=True)

part_file = next(temp_dir.glob("part-*.csv"))
export_path = silver_dir / "sales_silver.csv"
shutil.move(str(part_file), str(export_path))

shutil.rmtree(temp_dir)

after_count = clean.count()

print("Rows before preprocessing:", before_count)
print("Rows after preprocessing:", after_count)
print("Rows removed as exact duplicates:", before_count - after_count)
print("Silver export:", export_path)

Rows before preprocessing: 1000
Rows after preprocessing: 990
Rows removed as exact duplicates: 10
Silver export: data_lake/silver/preprocessed_sales/sales_silver.csv


In [10]:
silver_check = (spark.read
                .option("header", True)
                .option("inferSchema", True)
                .csv(str(silver_dir)))

print("Silver row count read back from disk:", silver_check.count())
silver_check.printSchema()
silver_check.show(10, truncate=False)

assert silver_check.count() == after_count
assert before_count - after_count == duplicate_count
print("Verification passed.")

Silver row count read back from disk: 990
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+----------------------+-----------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|order_id|customer_id|customer_name |product               |category   |quantity|unit_price|discount_percent|payment_method  |city     |state        |order_date|order_status|
+--------+-----------+--------------+----------------------+

## Preprocessing summary

The Bronze file was retained unchanged. Spark was used to calculate duplicate, missing-value, text-format, numeric-validation, and date-validation discrepancies. Exact duplicate rows were removed; string whitespace and obvious capitalization were standardized; blank descriptive fields were filled with Unknown; invalid quantity, unit-price, and discount values were replaced with valid-column medians; and recognizable dates were converted to a consistent date type. Unparseable dates were kept as null so that otherwise useful sales records were not discarded. The result was written to data_lake/silver/preprocessed_sales/ and exported as silver/sales_silver.csv.